# What is an AI agent?

The word "agent" is overloaded. In this notebook we mean specifically: **an LLM in a loop that can call tools.** That's the entire idea. Everything sophisticated downstream (planning, memory, multi-agent orchestration) is variations on this loop.

The loop, in pseudo-code:

```
while not done:
    response = llm(messages, tools)
    if response.tool_calls:
        for call in response.tool_calls:
            result = run_tool(call.name, call.arguments)
            messages.append(tool_result)
    else:
        done = True
return response.text
```

That's it. Six lines. We'll build it from scratch, then look at the production-ish version in `ai_playground.agents`.

**Why this pattern works.** An LLM by itself is a fixed-context text predictor — useful, but blind to current state (it doesn't know what files exist, what the weather is, what your DB has). Tools give it eyes and hands. The loop is what lets it *act* on intermediate results: read a file, decide what to do next, read another file, write an answer.

This pattern is called **ReAct** (Reasoning + Acting). [Paper: Yao et al., 2022](https://arxiv.org/abs/2210.03629).

See also: [docs/PAPERS.md § Agents](../../docs/PAPERS.md) · [Anthropic: Building effective agents](https://www.anthropic.com/research/building-effective-agents)

## 1. The simplest possible agent

We'll skip the LLM API at first and use a **scripted backend** — a fake model that returns pre-baked responses. This lets us focus on the loop itself, with no network and no surprises.

The "model" will pretend to answer "what is 17 * 23?" by asking us to call a calculator tool, then reading the result.

In [ ]:
import sys
sys.path.insert(0, '../../src')

from ai_playground.agents import Tool, ToolRegistry
from ai_playground.agents.llm import LLMResponse, ToolCall

# A tiny scripted 'LLM' — returns responses in order.
class ScriptedLLM:
    def __init__(self, responses):
        self.responses = list(responses)
    def complete(self, messages, tools=None, system=None, max_tokens=1024):
        print(f'  [scripted LLM called with {len(messages)} message(s)]')
        return self.responses.pop(0)

# A real tool — actually computes things.
calc = Tool(
    name='calculator',
    description='evaluate a math expression',
    input_schema={'type': 'object', 'properties': {'expr': {'type': 'string'}}, 'required': ['expr']},
    func=lambda expr: str(eval(expr, {'__builtins__': {}}, {})),
)

registry = ToolRegistry([calc])

# Pretend the model decides to call the tool, then reads the result.
script = [
    LLMResponse(
        text="Let me compute that.",
        tool_calls=[ToolCall(id='c1', name='calculator', arguments={'expr': '17 * 23'})],
        stop_reason='tool_use',
    ),
    LLMResponse(text="17 * 23 = 391.", stop_reason='end_turn'),
]
llm = ScriptedLLM(script)

## 2. Now the loop itself

This is the whole agent in ~15 lines. Read it carefully — every other agent framework you'll ever see is essentially this with more bells on.

In [ ]:
def minimal_agent(llm, tools, user_message, max_iter=5):
    messages = [{'role': 'user', 'content': user_message}]
    for i in range(max_iter):
        print(f'\n--- iteration {i} ---')
        response = llm.complete(messages, tools=tools.list())
        # Record the assistant's turn so the model sees its own tool requests
        messages.append({'role': 'assistant', 'content': [
            *([{'type': 'text', 'text': response.text}] if response.text else []),
            *[{'type': 'tool_use', 'id': c.id, 'name': c.name, 'input': c.arguments} for c in response.tool_calls],
        ]})
        if not response.tool_calls:
            print(f'  model says: {response.text}')
            return response.text
        # Dispatch each tool and feed results back
        results = []
        for call in response.tool_calls:
            print(f'  model wants to call {call.name}({call.arguments})')
            r = tools.dispatch(call.name, call.arguments, call.id)
            print(f'  -> tool returned: {r.content!r}')
            results.append(r.to_anthropic_block())
        messages.append({'role': 'user', 'content': results})
    return '[hit max iterations]'

answer = minimal_agent(llm, registry, 'what is 17 * 23?')
print(f'\nFINAL: {answer}')

## 3. The ReAct cycle in slow motion

Notice what just happened:

1. We sent the user's question to the "model".
2. Instead of answering, the model **acted**: it asked us to invoke `calculator` with `expr='17 * 23'`.
3. We ran the tool. Its output (`'391'`) went back to the model as a `tool_result` message.
4. Now the model had the missing piece, and on the next turn it **reasoned**: combined the original question with the tool output and produced a natural-language answer.

This Reason-Act-Reason-Act pattern is what makes an agent useful for anything beyond a single LLM call. The model uses tools to fill in what it doesn't know; the tool outputs constrain its next reasoning step.

Key insight: **"tool calling" isn't a special model capability.** The model just emits structured text (`tool_use` blocks), and a wrapper parses those out and dispatches. Anthropic's API hides the parsing inside the SDK; OpenAI's function calling does the same. Underneath, it's pattern-matching on tokens.

## 4. Same thing, with the real Agent class

The minimal loop above is essentially `ai_playground.agents.Agent.run()` with more error handling. Same behavior:

In [ ]:
from ai_playground.agents import Agent

# Reset the scripted LLM (we consumed its responses above)
llm = ScriptedLLM([
    LLMResponse(
        text="Let me compute that.",
        tool_calls=[ToolCall(id='c1', name='calculator', arguments={'expr': '17 * 23'})],
        stop_reason='tool_use',
    ),
    LLMResponse(text="17 * 23 = 391.", stop_reason='end_turn'),
])

agent = Agent(llm=llm, tools=ToolRegistry([calc]))
result = agent.run('what is 17 * 23?')
print(f'final text: {result.final_text}')
print(f'stop reason: {result.stop_reason}')
print(f'steps taken: {len(result.steps)}')
for i, step in enumerate(result.steps):
    print(f'  step {i}: text={step.response.text!r}, tool_calls={[c.name for c in step.response.tool_calls]}, results={[r.content for r in step.tool_results]}')

## 5. Now with a real LLM (optional — needs an API key)

Swap `ScriptedLLM` for `ClaudeBackend`. Set `ANTHROPIC_API_KEY` in your environment first. Install the agents extra: `uv sync --extra agents`.

The loop doesn't change. That's the point of the backend abstraction.

In [ ]:
# Uncomment to run with a real model:
# from ai_playground.agents import ClaudeBackend, builtin_tools
# 
# agent = Agent(
#     llm=ClaudeBackend(),
#     tools=ToolRegistry(builtin_tools(include_shell=False)),
#     system='You are a careful assistant. Use tools when needed.',
# )
# result = agent.run('Read /etc/hostname and tell me what host I am on.')
# print(result.final_text)
# for step in result.steps:
#     print(f' - {[c.name for c in step.response.tool_calls]} -> {[r.content[:60] for r in step.tool_results]}')

## Key takeaways

- **An agent is a loop.** Model emits text or tool calls; tool results feed back into the next model call; repeat until the model emits no more tool calls.
- **Tool calling is text.** The model emits a structured block; a parser dispatches; results come back as another message. There's no magic.
- **The backend abstraction matters.** Same loop, swap Claude for a local transformer for a different provider — the surface area is small on purpose.
- **Termination matters.** Always cap iterations. Models will happily loop forever.

### Next

- `01_tool_use.ipynb` — designing useful tools, JSON schemas, the security trade-offs of giving a model shell access
- `02_memory.ipynb` — short-term (conversation) and long-term (vector store) memory
- (later) planning, multi-agent orchestration